# Assignment 1: Introduction to Language Modeling

RNN-based autoregressive language model on Wikipedia paragraphs.

Structure:
- **Part 1**: tokenization (vocab + `A1Tokenizer`)
- **Part 2**: loading texts, batching
- **Part 3**: define the RNN model
- **Part 4**: training loop
- **Part 5**: evaluation (next-word prediction, perplexity, embedding neighbors)

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import math
import torch
import nltk

# First-time only: download the NLTK tokenizer data.
nltk.download('punkt_tab', quiet=True)
nltk.download('punkt', quiet=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# Paths to the text files from the skeleton archive. Adjust if needed.
TRAIN_FILE = 'wiki.train.txt'
VAL_FILE = 'wiki.valid.txt'

## Part 2 first: load the data

We need the texts to build the vocabulary, so let's load them before Part 1.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    'text',
    data_files={'train': TRAIN_FILE, 'val': VAL_FILE},
)

# Drop empty lines (paragraphs are separated by blanks in the source files).
dataset = dataset.filter(lambda x: x['text'].strip() != '')

print('train size:', len(dataset['train']))
print('val size:  ', len(dataset['val']))
print()
print('Example paragraph:')
print(dataset['train'][8]['text'][:400])

## Part 1: Tokenization

### Task 1.1 + 1.2: build the vocabulary

We use NLTK's `word_tokenize` on lowercased paragraphs, then take the most frequent words up to `max_voc_size` (minus 4 slots for the special symbols).

In [ ]:
from tokenizer import build_tokenizer, A1Tokenizer, SPECIAL_TOKENS

MAX_VOC_SIZE = 10000

# Builds the vocabulary AND wraps it in an A1Tokenizer in one step.
tokenizer = build_tokenizer(
    (item['text'] for item in dataset['train']),
    max_voc_size=MAX_VOC_SIZE,
)

print('vocab size:', len(tokenizer))
print('special tokens:', SPECIAL_TOKENS)
print('first 10 vocab entries:', list(tokenizer.str_to_int.items())[:10])

In [ ]:
# Sanity checks for Task 1.2:
assert len(tokenizer) <= MAX_VOC_SIZE
for s in SPECIAL_TOKENS:
    assert s in tokenizer.str_to_int, f'{s} missing from vocab'

# Common words should be in, rare words should not be.
for w in ['the', 'and', 'of', 'a']:
    assert w in tokenizer.str_to_int, f'{w} should be in vocab'
for w in ['cuboidal', 'epiglottis']:
    if w in tokenizer.str_to_int:
        print(f'note: rare word {w!r} is in vocab (may be OK depending on data)')

# Round-trip a word: string -> int -> string.
i = tokenizer.str_to_int['the']
assert tokenizer.int_to_str[i] == 'the'
print('sanity checks passed')

### Task 1.3: HuggingFace-like tokenizer

`A1Tokenizer.__call__` handles tokenization, padding, truncation, and returning torch tensors. Test it on a small batch where the inputs have different lengths.

In [ ]:
test_texts = ['This is a test.', 'Another test.']
out = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True)
print(out)

# Decode the first sequence back to words (skipping special tokens):
print('decoded[0]:', tokenizer.decode(out['input_ids'][0]))

In [ ]:
# Save the tokenizer so we don't need to rebuild the vocab next time.
tokenizer.save('a1_tokenizer.json')

# Load it back, just to confirm save/load works.
reloaded = A1Tokenizer.from_file('a1_tokenizer.json')
assert len(reloaded) == len(tokenizer)
print('tokenizer save/load OK')

## Part 2: batching (quick demo)

We don't include this in the submission, but let's quickly verify a `DataLoader` works as expected.

In [ ]:
from torch.utils.data import DataLoader

demo_loader = DataLoader(dataset['train'], batch_size=4, shuffle=False)
for batch in demo_loader:
    # HuggingFace datasets give us a dict of lists when batched naively.
    print('batch keys:', batch.keys())
    print('first text:', batch['text'][0][:120])
    break

## Part 3: define the RNN language model

### Task 3.1 + 3.2

In [ ]:
from model import A1RNNModelConfig, A1RNNModel

config = A1RNNModelConfig(
    vocab_size=len(tokenizer),
    embedding_dim=128,
    hidden_dim=256,
    num_layers=1,
    pad_token_id=tokenizer.pad_id,
)
model = A1RNNModel(config)
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f'\ntotal parameters: {n_params:,}')

In [ ]:
# Sanity check: a dummy 1 x N input should produce a 1 x N x V logits tensor.
N = 7
dummy = torch.randint(0, len(tokenizer), (1, N))
with torch.no_grad():
    out = model(dummy)
print('logits shape:', out['logits'].shape, '(expected: 1 x', N, 'x', len(tokenizer), ')')
assert out['logits'].shape == (1, N, len(tokenizer))

# Loss should also work if we pass labels.
labels = dummy.clone()
out = model(dummy, labels=labels)
print('loss with labels:', out['loss'].item())

## Part 4: train the model

Set `dev_mode = True` to train on a tiny subset for debugging. Switch to `False` for the real run.

In [ ]:
from torch.utils.data import Subset
from trainer import A1Trainer, A1TrainingArguments

dev_mode = False

if dev_mode:
    train_ds = Subset(dataset['train'], range(1000))
    val_ds = Subset(dataset['val'], range(200))
    epochs = 1
else:
    train_ds = dataset['train']
    val_ds = dataset['val']
    epochs = 3

args = A1TrainingArguments(
    output_dir='trainer_output',
    num_train_epochs=epochs,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    learning_rate=1e-3,
    max_seq_length=128,
    log_every=100,
)

trainer = A1Trainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

trainer.train()

## Part 5: Evaluation

### Task 5.1: predict the next word

Encode a prompt, take the logits at the position *before* the EOS token, and read off the top-k predictions.

In [ ]:
def predict_next(model, tokenizer, prompt, k=5):
    model.eval()
    enc = tokenizer(prompt, return_tensors='pt')
    input_ids = enc['input_ids'].to(next(model.parameters()).device)
    with torch.no_grad():
        out = model(input_ids)
    # Our tokenizer appends EOS, so the *prediction* for what follows the
    # last real word lives at position -2 (i.e. one before EOS).
    logits = out['logits'][0, -2]
    topk = torch.topk(logits, k)
    return [
        (tokenizer.int_to_str[i.item()], s.item())
        for i, s in zip(topk.indices, topk.values)
    ]

for prompt in [
    'She lives in San',
    'The president of the United',
    'He played the guitar and',
]:
    print(f'prompt: {prompt!r}')
    for word, score in predict_next(model, tokenizer, prompt):
        print(f'  {word:20s} {score:.3f}')
    print()

### Task 5.2: perplexity on the validation set

In [ ]:
val_loss, val_ppl = trainer.evaluate()
print(f'validation cross-entropy: {val_loss:.4f}')
print(f'validation perplexity:    {val_ppl:.2f}')

### Task 5.3: inspect the learned word embeddings

In [ ]:
import torch.nn as nn

def nearest_neighbors(emb, voc, inv_voc, word, n_neighbors=5):
    if word not in voc:
        print(f'{word!r} not in vocab')
        return []
    test_emb = emb.weight[voc[word]]
    sim = nn.CosineSimilarity(dim=1)(test_emb, emb.weight)
    top = sim.topk(n_neighbors + 1)
    # Skip index 0: it is the query word itself.
    return [
        (inv_voc[ix.item()], cos.item())
        for ix, cos in zip(top.indices[1:], top.values[1:])
    ]

for w in ['sweden', 'king', 'three', 'london', 'small']:
    print(f'neighbors of {w!r}:')
    for nbr, score in nearest_neighbors(
        model.embedding, tokenizer.str_to_int, tokenizer.int_to_str, w
    ):
        print(f'  {nbr:20s} {score:.3f}')
    print()

In [ ]:
# Optional: 2D PCA plot of selected embeddings.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD

def plot_embeddings_pca(emb, voc, words):
    words = [w for w in words if w in voc]
    vecs = np.vstack([
        emb.weight[voc[w]].detach().cpu().numpy() for w in words
    ])
    vecs -= vecs.mean(axis=0)
    twodim = TruncatedSVD(n_components=2).fit_transform(vecs)
    plt.figure(figsize=(6, 6))
    plt.scatter(twodim[:, 0], twodim[:, 1], c='r', edgecolors='k')
    for w, (x, y) in zip(words, twodim):
        plt.text(x + 0.02, y, w)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_embeddings_pca(
    model.embedding,
    tokenizer.str_to_int,
    [
        'sweden', 'denmark', 'europe', 'africa', 'london', 'stockholm',
        'large', 'small', 'great', 'black',
        'three', 'seven', 'ten',
    ],
)

## Reloading later

Once trained, you can skip training entirely:

```python
from model import A1RNNModel
from tokenizer import A1Tokenizer

tokenizer = A1Tokenizer.from_file('a1_tokenizer.json')
model = A1RNNModel.from_pretrained('trainer_output').to(device)
```